# Setup

In [ ]:
import gc
import os
import sys
import logging
import warnings
from pathlib import Path
import matplotlib as mpl
import pickle
import session_info

SRCDIR = Path('../../..')
HOMEDIR = SRCDIR / '..'
PLOTDIR = HOMEDIR / "plots" / "kennedi_xenium"
DATADIR = HOMEDIR / "data" / "processed" / "spatial" / "Xenium" / "kennedi_flu"
if not str(SRCDIR) in sys.path:
    sys.path.insert(0, str(SRCDIR))

logging.basicConfig(level="WARNING")
warnings.simplefilter("ignore", FutureWarning)
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", RuntimeWarning)
warnings.simplefilter("ignore", DeprecationWarning)

from single_cell.preprocess import *
from single_cell.plot import *
from single_cell.analysis import *
from spatial_seq.plot import *
from utils import *

import squidpy as sq
import cellcharter as cc
# import spatialdata as sd
# import spatialdata_plot as sdp

CORES = 20
%matplotlib inline
# R_preload()
mpl.rcdefaults()
plt.rcParams["figure.figsize"] = (8, 8)
gc.collect()

session_info.show()

In [ ]:
niche_key = "cellcharter_k12"
celltype_key = "LabelTransfer_OT"
sample1 = "Ctrl_D14_2"

# Squidpy metrics

In [ ]:
cc.tl.boundaries(adata, niche_key)
cc.pl.boundaries(adata, "Ctrl_D14_1", "Sample", niche_key)

### Composition

In [ ]:
cc.gr.enrichment(adata, niche_key, celltype_key)
cc.pl.enrichment(adata, niche_key, celltype_key,
                 group_cluster=False,
                 figsize=(12,8),
                 save=PLOTDIR / "niches" / f"CompositionEnrichment-{niche_key}.jpg")
cc.pl.enrichment(adata, niche_key, celltype_key,
                 group_cluster=True, label_cluster=True,
                 figsize=(12,8),
                 save=PLOTDIR / "niches" / f"CompositionEnrichment-{niche_key}-Clustered.jpg")

In [ ]:
# neighborhood composition
f = plt.figure(figsize=(20, 12), layout="constrained")
sf = f.subfigures(1,2,width_ratios=[1.3,2])
plot_cluster_stackedbarplot(adata, niche_key, celltype_key, pct=True, ax=sf[0].subplots(1, 1))
plot_cluster_stackedbarplot(adata, celltype_key, niche_key, pct=True, ax=sf[1].subplots(1, 1))
f.suptitle("Composition Barplots")
f.savefig(PLOTDIR / "niches" / f"CompositionBarplot-{niche_key}_celltype.jpg")

crosstab = pd.crosstab(adata.obs[celltype_key], adata.obs[niche_key]).astype(int)
f, axs = plt.subplots(1, 2, figsize=(25, 15), layout="constrained")
sns.heatmap(
    crosstab.div(crosstab.sum(axis=1), axis=0),
    cmap="Reds",
    annot=crosstab,
    fmt="d",
    ax=axs[0],
)
axs[0].set_title("Normalized by row (per celltype)", size=12)

sns.heatmap(
    crosstab.div(crosstab.sum(axis=0), axis=1),
    cmap="Reds",
    annot=crosstab,
    fmt="d",
    ax=axs[1],
)
axs[1].set_title("normalized by column (per neighborhood)", size=12)
f.suptitle("Percent Celltypes by Neighorhood\n(count annotated)", size=18)
f.savefig(PLOTDIR / "niches" / f"CompositionHeatmap-{niche_key}_celltype.jpg")

### Proximity Enrichment

In [ ]:
# proximity enrichment
RUN = False
if RUN is True: clear_uns(adata, "nhood")

for key in [niche_key, celltype_key]:
    if RUN is True:
        cc.gr.nhood_enrichment(adata, cluster_key=key, n_jobs=CORES, pvalues=True)
    cc.pl.nhood_enrichment(
        adata,
        cluster_key=key,
        annotate=True,
        vmin=-1,
        vmax=1,
        figsize=(5,5) if key==niche_key else (15,15),
        cmap="RdBu_r",
        title=f"Proximity enrichment ({key})",
        save=PLOTDIR / "niches" / f"ProximityEnrichment-{key}.jpg"
    )

    cc.pl.nhood_enrichment(
        adata,
        cluster_key=key,
        annotate=True,
        significance=5e-2,
        vmin=-1,
        vmax=1,
        figsize=(5,5) if key==niche_key else (15,15),
        cmap="RdBu_r",
        title=f"Proximity enrichment ({key}) significance",
        save=PLOTDIR / "niches" / f"ProximityEnrichment-{key}-significance.jpg"
    )

In [ ]:
# differential proximity enrichment
import cellcharter as cc
RUN = True
if RUN is True: clear_uns(adata, "nhood")

for key in [niche_key, celltype_key]:
    if RUN is True:
        cc.gr.diff_nhood_enrichment(
            adata, cluster_key=key, condition_key="Groups"
            n_jobs=CORES, pvalues=True, library_key="Sample")
    cc.pl.nhood_enrichment(
        adata,
        cluster_key=key,
        annotate=True,
        vmin=-1,
        vmax=1,
        figsize=(5,5) if key==niche_key else (15,15),
        cmap="RdBu_r",
        title=f"Differential Proximity enrichment ({key})",
        save=PLOTDIR / f"{niche_key}-DiffProximityEnrichment.jpg"
    )

### Co-Occurrence

In [ ]:
adata_samp1 = adata[adata.obs["Sample"] == sample1]
f,ax = plt.subplots(1,1, figsize=(25,10))
sc.pl.embedding(adata_samp1, "SPATIAL", color=niche_key, ax=ax, show=False, size=5)
for r in [100,250,500,1000,2000]:
    circle = plt.Circle((20000, 29000), r, color='black', fill=False, linewidth=1.5)
    ax.text(20000, 29000+r, r, fontsize=10, ha='center', va='bottom')
    ax.add_patch(circle)

In [ ]:
set_interval = np.arange(0, 500, 10)

for key in [niche_key, celltype_key]:
    sq.gr.co_occurrence(
        adata_samp1,
        cluster_key=key,
        spatial_key="SPATIAL",
        interval=set_interval,
        n_jobs=CORES,
    )

    sq.pl.co_occurrence(
        adata, cluster_key=key
    )

In [ ]:
# run = False
# if run is True:
#     warnings.filterwarnings("ignore")
#     split_adatas = {}
#     for sample in tqdm(adata.obs["sample"].unique()):
#         tmp = adata[adata.obs["sample"] == sample].copy()
#         sq.gr.co_occurrence(
#             tmp,
#             spatial_key="SPATIAL",
#             n_jobs=CORES,
#             interval=set_interval,
#             cluster_key="celltype",
#             show_progress_bar=False,
#         )
#         split_adatas[sample] = tmp

# SAMPLES = pd.Series(split_adatas.keys())

# for tp in adata.obs["timepoint"].cat.categories:
#     adata.uns['celltype_co_occurrence'] = {}
#     adata.uns['celltype_co_occurrence']['occ'] = np.add.reduce(
#         [split_adatas[sample].uns['celltype_co_occurrence']['occ'] for sample in SAMPLES[SAMPLES.str.contains(tp)]]
#     )
#     adata.uns['celltype_co_occurrence']['interval'] = set_interval

#     sq.pl.co_occurrence(
#         adata, cluster_key="celltype", clusters="Neu Phox2b+"
#     )

### Ripley's statistic (distribution)

In [ ]:
# sq.gr.ripley(adata, cluster_key=, mode=mode)

# Hotspot

# Save/Load

In [ ]:
# save
adata.write(DATADIR / "integrated-spatial_seq.h5ad")
adata

In [ ]:
# load
adata = sc.read_h5ad(DATADIR / "integrated.h5ad")
adata